<a href="https://colab.research.google.com/github/ofir2207/Cloud-project/blob/main/ex7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class IndexService:
    def __init__(self):
        self.documents = {}   # doc_id -> document dict
        self.index = {}       # word  -> set of doc_ids (inverted index)

    def add_document(self, doc_data):
        doc_id = str(len(self.documents) + 1)
        self.documents[doc_id] = {**doc_data, 'id': doc_id}
        words = doc_data['content'].lower().split()
        for word in words:
            if word not in self.index:
                self.index[word] = set()
            self.index[word].add(doc_id)
        return self.documents[doc_id]

    def get_document(self, doc_id):
        return self.documents.get(doc_id)

    def search_word(self, word):
        return list(self.index.get(word.lower(), set()))

    def word_count_in_doc(self, word, doc_id):
        # Count occurrences of word in doc (used for ranking)
        doc = self.documents.get(doc_id)
        if not doc:
            return 0
        return doc['content'].lower().split().count(word.lower())


In [ ]:
class QueryService:
    def __init__(self, index_service):
        self.index_service = index_service
        self.queries = {}

    def create_query(self, query_data):
        try:
            query_id = str(len(self.queries) + 1)
            search_terms = query_data['terms']
            operator = query_data.get('operator', 'AND').upper()

            results = set()
            for term in search_terms:
                doc_ids = set(self.index_service.search_word(term))
                if not results:
                    results = doc_ids
                elif operator == 'OR':
                    results |= doc_ids   # union
                else:                    # AND
                    results &= doc_ids   # intersection

            query = {
                'id': query_id,
                'terms': search_terms,
                'operator': operator,
                'results': list(results),
                'timestamp': query_data.get('timestamp', 'now')
            }
            self.queries[query_id] = query
            return query
        except Exception as e:
            return {'error': str(e)}


In [ ]:
class ResultService:
    def __init__(self, index_service, query_service):
        self.index_service = index_service
        self.query_service = query_service
        self.results = {}

    def _score(self, doc_id, terms):
        # Simple TF ranking: sum of term counts across all search terms
        return sum(self.index_service.word_count_in_doc(t, doc_id) for t in terms)

    def format_results(self, query_id):
        try:
            query = self.query_service.queries.get(query_id)
            if not query:
                return {'error': 'Query not found'}

            formatted = []
            for doc_id in query['results']:
                doc = self.index_service.get_document(doc_id)
                if doc:
                    formatted.append({
                        'doc_id': doc_id,
                        'title': doc['title'],
                        'snippet': doc['content'][:100] + '...',
                        'score': self._score(doc_id, query['terms'])
                    })

            # Sort by score descending – most relevant first
            formatted.sort(key=lambda x: x['score'], reverse=True)

            result_id = str(len(self.results) + 1)
            result = {
                'id': result_id,
                'query_id': query_id,
                'formatted_results': formatted,
                'count': len(formatted)
            }
            self.results[result_id] = result
            return result
        except Exception as e:
            return {'error': str(e)}


In [ ]:
class UserService:
    def __init__(self):
        self.users = {
            '1': {'id': '1', 'name': 'John Doe',  'email': 'john@example.com'},
            '2': {'id': '2', 'name': 'Jane Doe',  'email': 'jane@example.com'}
        }

    def get_user(self, user_id):
        return self.users.get(user_id, {})


In [ ]:
def main():
    index_service  = IndexService()
    query_service  = QueryService(index_service)
    result_service = ResultService(index_service, query_service)
    user_service   = UserService()

    docs = [
        {'title': 'Python Programming',
         'content': 'Python is a popular programming language for cloud computing and cloud services'},
        {'title': 'Cloud Services',
         'content': 'Cloud computing enables scalable microservices architecture'},
        {'title': 'Microservices Guide',
         'content': 'Microservices python pattern for scalable cloud cloud cloud applications'},
    ]
    for doc in docs:
        added = index_service.add_document(doc)
        print(f"Added doc {added['id']}: {added['title']}")

    print()
    print("=== AND query: ['cloud', 'python'] ===")
    q_and = query_service.create_query({'terms': ['cloud', 'python'], 'operator': 'AND'})
    r_and = result_service.format_results(q_and['id'])
    for item in r_and['formatted_results']:
        print(f"  [score={item['score']}] {item['title']}")

    print()
    print("=== OR query: ['cloud', 'python'] ===")
    q_or = query_service.create_query({'terms': ['cloud', 'python'], 'operator': 'OR'})
    r_or = result_service.format_results(q_or['id'])
    for item in r_or['formatted_results']:
        print(f"  [score={item['score']}] {item['title']}")

    print()
    print("=== UserService ===")
    print(user_service.get_user('1'))
    print(user_service.get_user('99'))  # not found -> {}


main()


In [ ]:
class IndexerFunction:
    def __init__(self):
        self.index = {}  # in real FaaS -> external storage (Redis / DynamoDB)

    def handle(self, event):
        doc_id  = event['document_id']
        content = event['content'].lower().split()
        for word in content:
            if word not in self.index:
                self.index[word] = set()
            self.index[word].add(doc_id)
        return {'status': 'success', 'indexed_words': len(content)}


class SearcherFunction:
    def __init__(self, index_service):
        self.index = index_service.index

    def handle(self, event):
        terms   = event['query'].lower().split()
        results = set()
        for term in terms:
            if term in self.index:
                results = self.index[term].copy() if not results else results & self.index[term]
        return {'status': 'success', 'results': list(results)}


class FaaSSimulator:
    def __init__(self):
        self.indexer     = IndexerFunction()
        self.searcher    = SearcherFunction(self.indexer)
        self.invocations = 0

    def invoke(self, function_name, event):
        self.invocations += 1
        if function_name == 'indexer':
            return self.indexer.handle(event)
        elif function_name == 'searcher':
            return self.searcher.handle(event)
        raise ValueError(f'Unknown function: {function_name}')


def demonstrate_faas():
    test_docs = [
        {'document_id': 'doc1', 'content': 'Python is a popular programming language for cloud computing'},
        {'document_id': 'doc2', 'content': 'Cloud computing enables scalable microservices architecture'}
    ]
    faas = FaaSSimulator()

    print('1. Event-Driven Invocation:')
    for doc in test_docs:
        result = faas.invoke('indexer', doc)
        print(f"  Indexed {doc['document_id']}: {result}")

    print('\n2. Independent Function Calls:')
    for q in [{'query': 'cloud computing'}, {'query': 'python programming'}]:
        result = faas.invoke('searcher', q)
        print(f"  '{q['query']}' -> {result}")

    print(f'\n3. Total invocations: {faas.invocations}')
    print('   In real FaaS: each invocation scales independently, billed per call.')


demonstrate_faas()
